Modelo XGBoost

In [1]:
# =========================================================================
# I. FASE DE PREPARACIÓN AVANZADA DE DATOS (Feature Engineering)
# =========================================================================
import pandas as pd
import re
import spacy 
import numpy as np
import sys 
import warnings
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS 
from sklearn.pipeline import FeatureUnion # Clave: combina features
from sklearn.base import BaseEstimator, TransformerMixin # Clave: transformadores custom

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 0. Configuración de SpaCy y Funciones ---
try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    
    def advanced_preprocess_spacy(text):
        """Aplica limpieza de ruido y lematización."""
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'@\w+|#\w+', '', text)
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\b\d+\b', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if not text: return ""
        doc = nlp(text)
        processed_tokens = [token.lemma_ for token in doc if token.is_alpha and len(token.text) > 2]
        return " ".join(processed_tokens)

except OSError:
    print("❌ ERROR CRÍTICO: SpaCy no está configurado.")
    sys.exit(1)

# --- Definición de Transformer Custom (Extracción de Features Numéricas) ---

class TextFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extrae características numéricas de la cadena de texto original."""
    
    def __init__(self):
        self.stop_words = set(ENGLISH_STOP_WORDS) 
        
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for text in X:
            text = str(text)
            length = len(text)
            caps_count = sum(1 for char in text if char.isupper()) / (length + 1e-6)
            punc_count = sum(1 for char in text if char in '?!.')
            words = text.lower().split()
            stop_count = sum(1 for word in words if word in self.stop_words) / (len(words) + 1e-6)
            
            features.append([length, caps_count, punc_count, stop_count])
        
        return np.array(features)

# --- 1. Carga, Limpieza Inicial y Preprocesamiento ---
print("\n--- PASO 1: CARGA Y PREPROCESAMIENTO ---")
try:
    df = pd.read_csv('../data/raw/youtoxic_english_1000.csv')
    df['Text'] = df['Text'].fillna('')
    
    # ⚠️ Lista completa y correcta de las 12 etiquetas
    global toxic_cols # Se define global para usarla en la Fase II
    toxic_cols = ['IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 
                  'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 
                  'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']
    
    for col in toxic_cols:
        if col not in df.columns: continue 
        if df[col].dtype == 'object': df[col] = df[col].astype(bool).astype(int) 
        elif df[col].dtype != 'int64': df[col] = df[col].astype(int)
    
    df['Text_Processed'] = df['Text'].apply(advanced_preprocess_spacy)
    
    print(f"✅ Preprocesamiento avanzado completado. Etiquetas a entrenar: {len(toxic_cols)}")

except FileNotFoundError:
    print("❌ ERROR CRÍTICO: ARCHIVO NO ENCONTRADO.")
    sys.exit(1)
except Exception as e:
    print(f"❌ ERROR CRÍTICO: Fallo en el preprocesamiento: {e}")
    sys.exit(1)


# --- 2. Separación de Datos y Pipeline de Features ---
print("\n--- PASO 2: SEPARACIÓN DE DATOS Y PIPELINE DE FEATURES ---")
X = df['Text_Processed']
Y = df['IsToxic']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y 
)
global Y_multi_train, Y_multi_test # Se definen globales para Fase II
Y_multi_train = df.iloc[X_train.index][toxic_cols] 
Y_multi_test = df.iloc[X_test.index][toxic_cols]

# 3. CREACIÓN DEL PIPELINE COMBINADO (FEATURE UNION)
pipeline_features = FeatureUnion([
    ('text_features', TfidfVectorizer(
        ngram_range=(1, 3), # Trigramas
        stop_words=list(ENGLISH_STOP_WORDS), 
        max_df=0.9, 
        min_df=3 
    )),
    ('num_features', TextFeatureExtractor())
])

# Se definen globales para Fase II
global X_train_vectorized, X_test_vectorized
X_train_vectorized = pipeline_features.fit_transform(X_train)
X_test_vectorized = pipeline_features.transform(X_test)

print(f"✅ Pipeline FeatureUnion completado. Características totales: {X_train_vectorized.shape[1]}")


--- PASO 1: CARGA Y PREPROCESAMIENTO ---
✅ Preprocesamiento avanzado completado. Etiquetas a entrenar: 12

--- PASO 2: SEPARACIÓN DE DATOS Y PIPELINE DE FEATURES ---
✅ Pipeline FeatureUnion completado. Características totales: 963


In [2]:
# =========================================================================
# II.B OPTIMIZACIÓN DEL MODELO XGBOOST CON OPTUNA (Preparación para Umbral)
# =========================================================================
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report
import xgboost as xgb 
import optuna 
import warnings
import numpy as np

# Se asume que X_train_vectorized, Y_multi_train, X_test_vectorized, Y_multi_test, y toxic_cols están definidos.

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb(trial):
    """Función objetivo para optimizar XGBoost (misma que antes)."""
    xgb_params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'use_label_encoder': False,
        'n_estimators': trial.suggest_int('n_estimators', 200, 700),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 8, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'gamma': trial.suggest_float('gamma', 1e-8, 0.5, log=True),
        'scale_pos_weight': trial.suggest_int('scale_pos_weight', 5, 50),
        'random_state': 42,
        'n_jobs': -1
    }

    base_clf = xgb.XGBClassifier(**xgb_params)
    model_ovr = OneVsRestClassifier(base_clf)
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model_ovr.fit(X_train_vectorized, Y_multi_train) 

    Y_pred = model_ovr.predict(X_test_vectorized)
    f1_micro = f1_score(Y_multi_test, Y_pred, average='micro', zero_division=0)
    
    return f1_micro

print("\n--- PASO 3.2: OPTIMIZANDO XGBOOST CON OPTUNA ---")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True) 

print(f"\n✅ XGBoost Optimización Optuna completada. Mejor F1: {study_xgb.best_value:.4f}")
print("Mejores parámetros XGBoost:")
print(study_xgb.best_params)

# --- Entrenamiento Final del Modelo Óptimo y Guardado de Probabilidades ---
print("\n--- PASO 3.2: ENTRENAMIENTO FINAL (Modelo XGBoost Optimizado) ---")

best_params_xgb = study_xgb.best_params
best_params_xgb.update({
    'objective': 'binary:logistic', 
    'eval_metric': 'logloss', 
    'use_label_encoder': False, 
    'random_state': 42, 
    'n_jobs': -1
})

base_clf_xgb_optimized = xgb.XGBClassifier(**best_params_xgb)
model_ovr_xgb_optimized = OneVsRestClassifier(base_clf_xgb_optimized)
model_ovr_xgb_optimized.fit(X_train_vectorized, Y_multi_train) 

# 🚨 CLAVE: Obtener las probabilidades de predicción
global Y_proba_optimized 
Y_proba_optimized = model_ovr_xgb_optimized.predict_proba(X_test_vectorized)

# Evaluación inicial (con umbral 0.5 por defecto)
Y_pred_default = model_ovr_xgb_optimized.predict(X_test_vectorized)
f1_micro_default = f1_score(Y_multi_test, Y_pred_default, average='micro', zero_division=0)

print("\n--- RESULTADOS BASE DEL MODELO OPTIMIZADO (Umbral 0.5) ---")
print(f"Micro F1-Score del Modelo (Default 0.5): {f1_micro_default:.4f}")

/Users/aroamateogomez/Desktop/BootcampIA/Proyecto10/Proyecto_10_X/proyecto10-grupo5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- PASO 3.2: OPTIMIZANDO XGBOOST CON OPTUNA ---


  0%|          | 0/100 [00:00<?, ?it/s]0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Best trial: 85. Best value: 0.576105: 100%|██████████| 100/100 [26:30<00:00, 15.90s/it]



✅ XGBoost Optimización Optuna completada. Mejor F1: 0.5761
Mejores parámetros XGBoost:
{'n_estimators': 661, 'learning_rate': 0.012442193110553983, 'max_depth': 13, 'subsample': 0.8923861259924781, 'colsample_bytree': 0.6253248043523429, 'gamma': 0.0001274187688307218, 'scale_pos_weight': 8}

--- PASO 3.2: ENTRENAMIENTO FINAL (Modelo XGBoost Optimizado) ---

--- RESULTADOS BASE DEL MODELO OPTIMIZADO (Umbral 0.5) ---
Micro F1-Score del Modelo (Default 0.5): 0.5761


In [3]:
# ================================================================================
# III. FASE DE OPTIMIZACIÓN DE UMBRAL POR ETIQUETA (Per-Label Threshold Tuning)
# ================================================================================
from sklearn.metrics import f1_score
import numpy as np

print("\n--- PASO 4: OPTIMIZACIÓN DEL UMBRAL POR ETIQUETA ---")

# Almacena los umbrales óptimos para cada clase
optimal_thresholds = {}
Y_pred_thresholded = np.zeros(Y_multi_test.shape) # Matriz de predicciones final

for i, label in enumerate(toxic_cols):
    
    # 1. Obtener las probabilidades y las etiquetas verdaderas para esta clase
    y_true = Y_multi_test.iloc[:, i]
    y_proba = Y_proba_optimized[:, i]
    
    best_f1 = 0
    best_thresh = 0.5 # Umbral inicial

    # 2. Iterar sobre un rango fino de posibles umbrales (0.01 a 0.5)
    # No tiene sentido ir más allá de 0.5 ya que estamos tratando con clases raras
    for thresh in np.arange(0.01, 0.5, 0.01): 
        # Convertir probabilidades a predicciones binarias usando el umbral
        y_pred_current = (y_proba >= thresh).astype(int)
        
        # Calcular el F1-Score
        f1 = f1_score(y_true, y_pred_current, zero_division=0)
        
        # Actualizar si encontramos un F1 mejor
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    optimal_thresholds[label] = best_thresh
    
    # Aplicar el umbral óptimo a la matriz de predicciones final
    Y_pred_thresholded[:, i] = (y_proba >= best_thresh).astype(int)
    
    print(f"[{label:18s}]: Mejor F1: {best_f1:.4f}, Umbral óptimo: {best_thresh:.2f}")


# --- Evaluación Final (Umbral Optimizado) ---

f1_micro_optimized_thresh = f1_score(Y_multi_test, Y_pred_thresholded, average='micro', zero_division=0)

print("\n--- RESULTADOS FINALES (XGBOOST + OPTUNA + UMBRAL OPTIMIZADO) ---")
print(f"Micro F1-Score del Modelo (¡MEJORADO!): {f1_micro_optimized_thresh:.4f}")
print("\nInforme de Clasificación:")
print(classification_report(Y_multi_test, Y_pred_thresholded, target_names=toxic_cols, zero_division=0))
print("\nUmbrales Óptimos por Etiqueta:")
print(pd.Series(optimal_thresholds).sort_values(ascending=False))


--- PASO 4: OPTIMIZACIÓN DEL UMBRAL POR ETIQUETA ---
[IsToxic           ]: Mejor F1: 0.7059, Umbral óptimo: 0.48
[IsAbusive         ]: Mejor F1: 0.5953, Umbral óptimo: 0.24
[IsThreat          ]: Mejor F1: 0.2667, Umbral óptimo: 0.03
[IsProvocative     ]: Mejor F1: 0.3262, Umbral óptimo: 0.13
[IsObscene         ]: Mejor F1: 0.5263, Umbral óptimo: 0.17
[IsHatespeech      ]: Mejor F1: 0.5455, Umbral óptimo: 0.40
[IsRacist          ]: Mejor F1: 0.4889, Umbral óptimo: 0.47
[IsNationalist     ]: Mejor F1: 0.0000, Umbral óptimo: 0.50
[IsSexist          ]: Mejor F1: 0.0000, Umbral óptimo: 0.50
[IsHomophobic      ]: Mejor F1: 0.0000, Umbral óptimo: 0.50
[IsReligiousHate   ]: Mejor F1: 0.8000, Umbral óptimo: 0.02
[IsRadicalism      ]: Mejor F1: 0.0000, Umbral óptimo: 0.50

--- RESULTADOS FINALES (XGBOOST + OPTUNA + UMBRAL OPTIMIZADO) ---
Micro F1-Score del Modelo (¡MEJORADO!): 0.5613

Informe de Clasificación:
                 precision    recall  f1-score   support

        IsToxic       0.58 